In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
from pyspark.sql.functions import *
silver_df = (
    spark.readStream
    .format("delta")
    .load(
        "abfss://silver@travelappprojectstorage.dfs.core.windows.net/geofence/geofence_alert_clean"
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "alert_category",
    when(
        col("Message") == "Entered into Testing Zone",
        "TEST_ALERT"
    ).otherwise("REAL_ALERT")
)
silver_df = silver_df.withColumn(
    "alert_status",
    when(col("IsResolved") == True, "RESOLVED")
    .otherwise("UNRESOLVED")
)


In [0]:
silver_df = silver_df.withColumn(
    "resolution_minutes",
    when(
        col("IsResolved") == True,
        (
            unix_timestamp(col("ResolvedAt")) -
            unix_timestamp(col("CreatedAt"))
        ) / 60
    )
)

In [0]:
real_alert_df = silver_df.filter(
    col("alert_category") == "REAL_ALERT"
)


test_alert_df = silver_df.filter(
    col("alert_category") == "TEST_ALERT"
)


In [0]:
high_risk_df = real_alert_df.groupBy(
    "TouristId",
    "Nationality",
    "agency_name"
).agg(
    count("*").alias("total_alerts"),
    sum("risk_score").alias("total_risk_score"),
    max("CreatedAt").alias("latest_alert_time")
)

hotspot_df = real_alert_df.groupBy(
    "PlaceId",
    "danger_place_name"
).agg(
    count("*").alias("alert_count"),
    approx_count_distinct("TouristId").alias("unique_tourists"),
    avg("DistanceMeters").alias("avg_distance")
)

In [0]:
status_df = real_alert_df.groupBy(
    "alert_status"
).agg(
    count("*").alias("alert_count")
)

open_alert_df = real_alert_df.filter(
    col("alert_status") == "UNRESOLVED"
)


In [0]:
employee_resolution_df = real_alert_df.filter(
    col("IsResolved") == True
).withColumn(
    "resolved_employee",
    when(
        col("ResolvedByEmployeeId").isNull(),
        "UNKNOWN"
    ).otherwise(
        col("ResolvedByEmployeeId").cast("string")
    )
).groupBy(
    "resolved_employee"
).agg(
    count("*").alias("resolved_alert_count"),
    avg("resolution_minutes").alias("avg_resolution_minutes")
)

In [0]:
trend_df = real_alert_df.groupBy(
    window(col("CreatedAt"), "10 minutes")
).agg(
    count("*").alias("alerts_per_window")
).select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    col("alerts_per_window")
)

In [0]:
test_monitor_df = test_alert_df.groupBy(
    window(col("CreatedAt"), "10 minutes")
).agg(
    count("*").alias("test_alert_count")
).select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    col("test_alert_count")
)

In [0]:
checkpoint = "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/geofence_gold"
gold = "abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence"


In [0]:
high_risk_df.writeStream.format("delta").outputMode("complete") \
.option("checkpointLocation", f"{checkpoint}/high_risk") \
.trigger(availableNow=True)\
.start(f"{gold}/gold_high_risk_tourists")

hotspot_df.writeStream.format("delta").outputMode("complete") \
.option("checkpointLocation", f"{checkpoint}/hotspots") \
.trigger(availableNow=True)\
.start(f"{gold}/gold_danger_zone_hotspots")

status_df.writeStream.format("delta").outputMode("complete") \
.option("checkpointLocation", f"{checkpoint}/status_metrics") \
.trigger(availableNow=True)\
.start(f"{gold}/gold_alert_status_metrics")

open_alert_df.writeStream.format("delta").outputMode("append") \
.option("checkpointLocation", f"{checkpoint}/open_alerts") \
.trigger(availableNow=True)\
.start(f"{gold}/gold_open_alerts")

employee_resolution_df.writeStream.format("delta").outputMode("complete") \
.option("checkpointLocation", f"{checkpoint}/employee_resolution") \
.trigger(availableNow=True)\
.start(f"{gold}/gold_employee_resolution_metrics")

trend_df.writeStream.format("delta").outputMode("complete") \
.option("checkpointLocation", f"{checkpoint}/live_trends") \
.trigger(availableNow=True)\
.start(f"{gold}/gold_live_alert_trends")

test_monitor_df.writeStream.format("delta").outputMode("complete") \
.option("checkpointLocation", f"{checkpoint}/test_monitoring") \
.trigger(availableNow=True)\
.start(f"{gold}/gold_test_alert_monitoring")


In [0]:
gold_base = "abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence"

high_risk_df = spark.read.format("delta").load(
    f"{gold_base}/gold_high_risk_tourists"
)

hotspot_df = spark.read.format("delta").load(
    f"{gold_base}/gold_danger_zone_hotspots"
)

status_df = spark.read.format("delta").load(
    f"{gold_base}/gold_alert_status_metrics"
)

open_alert_df = spark.read.format("delta").load(
    f"{gold_base}/gold_open_alerts"
)

employee_resolution_df = spark.read.format("delta").load(
    f"{gold_base}/gold_employee_resolution_metrics"
)

trend_df = spark.read.format("delta").load(
    f"{gold_base}/gold_live_alert_trends"
)

test_monitor_df = spark.read.format("delta").load(
    f"{gold_base}/gold_test_alert_monitoring"
)

display(high_risk_df)
display(hotspot_df)
display(status_df)
display(open_alert_df)
display(employee_resolution_df)
display(trend_df)
display(test_monitor_df)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS default.gold_high_risk_tourists
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence/gold_high_risk_tourists';

CREATE TABLE IF NOT EXISTS default.gold_danger_zone_hotspots
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence/gold_danger_zone_hotspots';

CREATE TABLE IF NOT EXISTS default.gold_alert_status_metrics
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence/gold_alert_status_metrics';

CREATE TABLE IF NOT EXISTS default.gold_open_alerts
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence/gold_open_alerts';

CREATE TABLE IF NOT EXISTS default.gold_employee_resolution_metrics
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence/gold_employee_resolution_metrics';

CREATE TABLE IF NOT EXISTS default.gold_live_alert_trends
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence/gold_live_alert_trends';

CREATE TABLE IF NOT EXISTS default.gold_test_alert_monitoring
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/geofence/gold_test_alert_monitoring';

SHOW TABLES

In [0]:
%sql
DROP TABLE IF EXISTS default.gold_employee_resolution_metrics;
DROP TABLE IF EXISTS default.gold_live_alert_trends;
DROP TABLE IF EXISTS default.gold_test_alert_monitoring;